In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import os
path_project = os.path.dirname(os.path.abspath('.'))
dataset_path = os.path.join(path_project, 'dataset')

## Dataset

Credit Card Fraud Detection
- URL https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
- reference code https://www.kaggle.com/code/abubakaryagob/credit-card-fraud-detection-with-pytorch

In [2]:
df = pd.read_csv(os.path.join(dataset_path, 'creditcard', 'creditcard.csv'))
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# split data into training and testing

X = df.drop("Class", axis=1)
y = df["Class"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y) # stratify keeps class balance

# scale the values of x (better training)
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
# create tensor datasets from df
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.FloatTensor(y_train.values)
y_test = torch.FloatTensor(y_test.values)

In [5]:
percentage = 0.1
target_label = 0
target_X_train = X_train[y_train == target_label]
target_y_train = y_train[y_train == target_label]
non_target_X_train = X_train[y_train != target_label]
non_target_y_train = y_train[y_train != target_label]
limit_size = int(len(target_X_train) * percentage)
selected_indices = np.random.choice(len(target_X_train), size=limit_size, replace=False)
target_X_train = target_X_train[selected_indices]
target_y_train = target_y_train[selected_indices]
X_train = torch.concatenate([non_target_X_train, target_X_train])
y_train = torch.concatenate([non_target_y_train, target_y_train])
train_dataset = TensorDataset(X_train, y_train)

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [7]:
# create dataloaders
batch_size = 128
train_data_loader = DataLoader(train_dataset, batch_size=batch_size)

In [8]:
# Network Architecture
class FraudNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=4):
        super().__init__()
        self.input = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        # make the number of hidden dim layers configurable
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers.append(nn.ReLU())
        
        # final layer
        self.fc = nn.Linear(hidden_dim, 2)
        
    def forward(self, x):
        out = self.input(x)
        for layer in self.layers:
            out = layer(out)
        return self.fc(out)
    
class ImprovedFraudNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=4):
        super().__init__()
        self.input = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim)
        )
        # make the number of hidden dim layers configurable
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.BatchNorm1d(hidden_dim))
            self.layers.append(nn.Dropout(0.5))
        
        # final layer
        self.fc = nn.Linear(hidden_dim, 2)
        
    def forward(self, x):
        out = self.input(x)
        for layer in self.layers:
            out = layer(out)
        return self.fc(out)
    
class PrivateImprovedFraudNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=4):
        super().__init__()
        self.input = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.GroupNorm(1, hidden_dim)
        )
        # make the number of hidden dim layers configurable
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.GroupNorm(1, hidden_dim))
            self.layers.append(nn.Dropout(0.5))
        
        # final layer
        self.fc = nn.Linear(hidden_dim, 2)
        
    def forward(self, x):
        out = self.input(x)
        for layer in self.layers:
            out = layer(out)
        return self.fc(out)
    

In [9]:
# training function
def train_model(model, epochs, loss_fn, optimizer, train_data_loader):
    model.train()   
    for epoch in range(epochs):
        with tqdm(train_data_loader, unit="batch") as tepoch:
            for data, target in tepoch:
                data, target = data.to(device), target.to(device)
                tepoch.set_description(f"Epoch {epoch}")
                optimizer.zero_grad()
                preds = model(data)
                target = target.long()
                loss = loss_fn(preds, target)
                loss.backward()
                optimizer.step()
                tepoch.set_postfix(loss=loss.item())

In [10]:
inp_size = X_train.shape[1]
# model = FraudNet(inp_size, inp_size).to(device)
model = ImprovedFraudNet(inp_size, inp_size).to(device)
loss = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(),  lr = 1e-2)

In [ ]:
epochs = 50
train_model(model, epochs, loss, optim)

In [ ]:
model.eval()
preds = model(X_test.to(device)).argmax(dim=1)
print(classification_report(y_test, preds.cpu()))
roc_auc_score(y_test, preds.cpu())

In [11]:
inp_size = X_train.shape[1]
# model = FraudNet(inp_size, inp_size).to(device)
model = PrivateImprovedFraudNet(inp_size, inp_size).to(device)
loss = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(),  lr = 1e-2)

In [12]:
epochs = 50
train_model(model, epochs, loss, optim, train_data_loader)

Epoch 49: 100%|██████████| 159/159 [00:00<00:00, 212.05batch/s, loss=0.00319]


In [13]:
model.eval()
preds = model(X_test.to(device)).argmax(dim=1)
print(classification_report(y_test, preds.cpu()))
roc_auc_score(y_test, preds.cpu())

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     85295
         1.0       0.85      0.80      0.83       148

    accuracy                           1.00     85443
   macro avg       0.92      0.90      0.91     85443
weighted avg       1.00      1.00      1.00     85443



0.9019039248522218